In [2]:
import keras
import keras_tuner
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import linear_model
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error

/Users/benedikt/Documents/Master/SoSe 25/Deep Learning for Text Analytics/Coding/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Reading the given datasets and Scaling

In [3]:
# x_test = pd.read_csv('./resources/test_X.csv')
# x_test = x_test.drop(columns=[x_test.columns[0]])

x_train_raw = pd.read_csv('./resources/train_X.csv')
x_train_raw = x_train_raw.drop(columns=[x_train_raw.columns[0]])

y_train_raw = pd.read_csv('./resources/train_y.csv')
y_train_raw = y_train_raw.drop(columns=[y_train_raw.columns[0]])



# n_train = int(x_train_raw.shape[0]*0.7)

# x_train, x_val = x_train_raw.iloc[:n_train,:], x_train_raw.iloc[n_train:,:]
# y_train, y_val = y_train_raw.iloc[:n_train], y_train_raw.iloc[n_train:]

# Try random split and compare
x_train, x_val, y_train, y_val = train_test_split(x_train_raw, 
                                                  y_train_raw, 
                                                  test_size=0.3, 
                                                  random_state=42,
                                                  shuffle = True)


## Scaling

In [4]:

x_scaler = StandardScaler()
x_scaler.fit(x_train)

x_train_scaled, x_val_scaled = x_scaler.transform(x_train), x_scaler.transform(x_val)

'''
scaling the y_train. Keep the scaler for 'descaling' the y_pred
y_pred_standard_unscaled = y_scaler_standard.inverse_transform(y_pred_standard_scaled.reshape(-1, 1))
'''
y_scaler = StandardScaler()
y_scaler.fit(y_train)

y_train_scaled, y_val_scaled = y_scaler.transform(y_train), y_scaler.transform(y_val)

# Linear Regression and Tree based model

In [4]:
'''
linear regression
'''

lin_regr = linear_model.LinearRegression()
lin_regr.fit(x_train_scaled, y_train_scaled)

pred_val_lin_regr = lin_regr.predict(x_val_scaled)
# pred_val_lin_regr_unscaled = y_scaler.inverse_transform(pred_val_lin_regr.reshape(-1, 1))

# MSE_lin_regr = round(mean_squared_error(y_true=y_val, y_pred=pred_val_lin_regr_unscaled.flatten()),5)
MSE_lin_regr = round(mean_squared_error(y_true = y_val_scaled.flatten(), 
                                        y_pred = pred_val_lin_regr.flatten()),5)


'''
tree-based regression : XGBoost Classifier
'''
xgb_regr = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    eval_metric='rmse',
    n_jobs=1 
)

xgb_regr.fit(x_train_scaled, y_train_scaled)

pred_val_xgb_regr = xgb_regr.predict(x_val_scaled)
# pred_val_xgb_regr_unscaled = y_scaler.inverse_transform(pred_val_xgb_regr.reshape(-1, 1))

'''
rescaling the y_pred
'''

# MSE_xgb_regr = round(mean_squared_error(y_true=y_val, y_pred=pred_val_xgb_regr_unscaled.flatten()),5)
MSE_xgb_regr = round(mean_squared_error(y_true = y_val_scaled.flatten(), 
                                        y_pred = pred_val_xgb_regr.flatten()),5)

print('MSE of Linear Regression: {} and compared to MSE of Tree based model: {}'.format(MSE_lin_regr, MSE_xgb_regr))


MSE of Linear Regression: 0.01562 and compared to MSE of Tree based model: 0.00462


# Untuned Neural Network

## Global functions

In [5]:

def build_tf_model(n_features:int,
                   list_units:list,
                   dropout_rate:float,
                   activation:str)->tf.keras.Model:
    
    input_layer = tf.keras.layers.Input(shape=n_features)
    hidden_layers = input_layer

    for unit in list_units:
        hidden_layers = tf.keras.layers.Dense(units=unit, 
                                              activation=activation,
                                              kernel_initializer = (tf.keras.initializers.HeNormal() if activation=='relu' else tf.keras.initializers.GlorotNormal())
                                              )(hidden_layers)
        
        hidden_layers = tf.keras.layers.Dropout(rate=dropout_rate
                                                )(hidden_layers)

    output_layer = tf.keras.layers.Dense(units = 1,
                                         activation = 'linear'
                                         )(hidden_layers) 

    tf_model = tf.keras.Model(inputs = input_layer,
                              outputs = output_layer) 

    return tf_model


    
def training(model:tf.keras.Model,
             x_batch_data:tf.Tensor,
             y_batch_data:tf.Tensor,
             optimizer:None = None)->tf.Tensor:
    
    with tf.GradientTape() as tape:
        pred = model(x_batch_data, training=True)
        train_loss = tf.keras.losses.MSE(y_pred = tf.squeeze(pred,axis=-1),
                                         y_true = tf.squeeze(y_batch_data,axis=-1))
    
    # weight updates for backward pass
    gradients = tape.gradient(train_loss, model.trainable_variables) # compute weight updates for backward pass
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))#apply computed gradients
    
    return train_loss



## Batching the dataset

In [6]:
batch_size=32

#Splitting train and test dataset into mini-batches:
tf_train_rescaled = tf.data.Dataset.from_tensor_slices((x_train_scaled,y_train_scaled)).batch(batch_size)
tf_val_rescaled = tf.data.Dataset.from_tensor_slices((x_val_scaled,y_val_scaled)).batch(batch_size)


## Main part

In [7]:
# Tuning n_units(hidden layer), dropout_rate, and activation function
tf_model = build_tf_model(n_features = (x_train_scaled.shape[1], ),
                          list_units = [100, 50, 25],
                          dropout_rate = 0.3,
                          activation = 'relu')

# Tuning learning rate, optimizer and epochs
tf_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
n_epochs = 100

train_loss = []
val_loss = []

# Early stoppping criterium
patience = 5
wait = 0
# We would like to minimize our loss, thus we set the initial best value to inf+
best = float('inf')

for epoch in range(n_epochs):
    epoch_train_loss = []
    epoch_val_loss = []

    # Training
    for (x_batch_train,y_batch_train) in tf_train_rescaled:
        curr_train_loss = training(model = tf_model,
                              x_batch_data = x_batch_train,
                              y_batch_data = y_batch_train,
                              optimizer = tf_optimizer)

        epoch_train_loss.append(curr_train_loss.numpy())

    train_loss.append(np.mean(epoch_train_loss))

    # Validation 
    for (x_batch_val,y_batch_val) in tf_val_rescaled:

        predictions_val = tf_model(x_batch_val,training=False)
        
        curr_val_loss = tf.keras.losses.MSE(y_pred = tf.squeeze(predictions_val,axis=-1),
                                            y_true = tf.squeeze(y_batch_val,axis=-1))
        
        epoch_val_loss.append(curr_val_loss.numpy())

    epoch_val_loss = np.mean(epoch_val_loss)
    val_loss.append(epoch_val_loss)

    #Early stopping callback:
    wait += 1

    if epoch_val_loss < best:
      best = epoch_val_loss
      wait = 0
    if wait >= patience:
      break

print('MSE training: {} and compared to MSE validation: {}'.format(min(val_loss), min(train_loss)))


2025-05-24 14:37:17.096344: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-05-24 14:37:17.122937: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-05-24 14:37:17.588290: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-05-24 14:37:18.471216: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-05-24 14:37:20.216968: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-05-24 14:37:23.701177: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


MSE training: 0.17194749414920807 and compared to MSE validation: 0.17934483289718628


# Tuned Neural Network

## global function

In [8]:
def build_tf_model(n_features:int,
                   list_units:list,
                   dropout_rate:float,
                   activation:str,
                   regularization:list)->tf.keras.Model:
    
    input_layer = tf.keras.layers.Input(shape=n_features)
    hidden_layers = input_layer

    for unit in list_units:
        hidden_layers = tf.keras.layers.Dense(units=unit, 
                                              activation=activation,
                                              kernel_initializer = (tf.keras.initializers.HeNormal() if activation=='relu' else tf.keras.initializers.GlorotNormal()),
                                              kernel_regularizer= choose_regularization(regularization)
                                              )(hidden_layers)
        
        hidden_layers = tf.keras.layers.Dropout(rate=dropout_rate
                                                )(hidden_layers)

    output_layer = tf.keras.layers.Dense(units = 1,
                                         activation = 'linear'
                                         )(hidden_layers) 

    tf_model = tf.keras.Model(inputs = input_layer,
                              outputs = output_layer) 

    return tf_model


# def choose_regularization(regularization:list):

#     if len(regularization) == 1:
#         reg = regularization[0]

#         if reg[0] == 'l1':
#             l1_reg = reg[1]
#             return tf.keras.regularizers.L1(l1=l1_reg)
#         elif reg[0] == 'l2':
#             l2_reg = reg[1]
#             return tf.keras.regularizers.L2(l2=l2_reg)
#         else:
#             raise ValueError('Regularization not supported')        
        
#     elif len(regularization) == 2:
#         idx_1 = regularization[0]
#         idx_2 = regularization[1]

#         if idx_1[0] == 'l1' and idx_2[0] == 'l2':
#             l1_reg = idx_1[0][1]
#             l2_reg = idx_1[1][1]

#         elif idx_1[0] == 'l2' and idx_2[0] == 'l1':
#             l2_reg = idx_1[0][1]
#             l1_reg = idx_2[1][1]

#         else:
#             raise ValueError('Regularization not supported')
        
#         return tf.keras.regularizers.L1L2(l1=l1_reg, l2=l2_reg)
#     else:
#         raise ValueError('Regularization not supported')
    


def training(model:tf.keras.Model,
             x_batch_data:tf.Tensor,
             y_batch_data:tf.Tensor,
             optimizer:None = None)->tf.Tensor:
    
    with tf.GradientTape() as tape:
        pred = model(x_batch_data, training=True)
        train_loss = tf.keras.losses.MSE(y_pred = tf.squeeze(pred,axis=-1),
                                         y_true = tf.squeeze(y_batch_data,axis=-1))
        # reg_loss = tf.reduce_sum(model.losses)
        # total_loss = train_loss+reg_loss
    
    # weight updates for backward pass
    gradients = tape.gradient(train_loss, model.trainable_variables) # compute weight updates for backward pass
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))#apply computed gradients
    
    return train_loss

## Hyperparameter tuning without bathcing

the parameters are:
- Activation
- Dropout rate
- Regularization
- Number of units per layer
- Optimizer
- Learning rate


In [8]:
def keras_hp_optimization(hp:keras_tuner.HyperParameters)->keras.Sequential:

    # Activation
    curr_activation = hp.Choice('Activation', ['linear','relu', 'tanh', 'sigmoid'])
    
    # Dropout rate
    curr_dropout_rate = hp.Float('Dropout_rate',
                                    min_value = 0.0,
                                    max_value = 0.5,
                                    step = 0.05)
    
    # Regularization
    curr_regularization = hp.Choice('Regularization', ['l1', 'l2', 'l1_l2'])
    curr_regularization_value = hp.Float('regularization_value',   
                                        min_value = 0.0001,
                                        max_value = 0.001,
                                        step = 0.0001)
    
    if curr_regularization == 'l1':
        regularizer = keras.regularizers.L1(l1=curr_regularization_value)
    elif curr_regularization == 'l2':
        regularizer = keras.regularizers.L2(l2=curr_regularization_value)
    else:
        regularizer = keras.regularizers.L1L2(l1=curr_regularization_value, 
                                            l2=curr_regularization_value)

    model = keras.Sequential()

    max_hidden_layers = 3
    keep_sampling_hidden_units = True
    nr_hidden_layer = 0

    while keep_sampling_hidden_units:
        
        # Unit for hidden layer
        if nr_hidden_layer == 0:
            current_hidden_units = hp.Int('Units_hidden_layer{}'.format(nr_hidden_layer),
                                        min_value = 10,
                                        max_value = 200,
                                        step = 10)
        else:
            current_hidden_units = hp.Int('Units_hidden_layer{}'.format(nr_hidden_layer),
                                        min_value = 0,
                                        max_value = 200,
                                        step = 10)
        
        
        if current_hidden_units != 0:
            model.add(keras.layers.Dense(units = current_hidden_units,
                                        kernel_regularizer=regularizer,
                                        activation = curr_activation))
            
            model.add(keras.layers.Dropout(rate = curr_dropout_rate))

        if current_hidden_units == 0 or nr_hidden_layer == max_hidden_layers-1:
            keep_sampling_hidden_units = False
            
            
        
        nr_hidden_layer += 1
    
    model.add(keras.layers.Dense(units = 1,
                                activation = 'linear'))
    

    # Optimizer and learning rate
    current_optimizer = hp.Choice('optimizer', ['adam', 'sgd'])
    current_learning_rate = hp.Float('learning_rate',
                                    min_value = 0.0001,
                                    max_value = 0.001,
                                    step = 0.0001)
    model.compile(optimizer = (keras.optimizers.Adam(learning_rate = current_learning_rate) if current_optimizer == 'adam' else keras.optimizers.SGD(learning_rate = current_learning_rate)),
                            loss = 'mse')
    
    return model




es_callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    mode='min')

tuner = keras_tuner.RandomSearch(keras_hp_optimization,
                                 objective = 'val_loss',
                                 max_trials = 100,
                                 directory = './tmp/checkpoint',
                                 overwrite = True)


tuner.search(x_train_scaled, 
             y_train_scaled, 
             epochs = 100, 
             validation_data = (x_val_scaled, y_val_scaled), 
             callbacks = [es_callback])

best_model = tuner.get_best_models()[0]

print('Best hyperparameters: {}'.format(tuner.get_best_hyperparameters()[0].values))


Trial 100 Complete [00h 00m 07s]
val_loss: 0.08278972655534744

Best val_loss So Far: 0.00665176659822464
Total elapsed time: 00h 10m 50s
Best hyperparameters: {'Activation': 'relu', 'Dropout_rate': 0.0, 'Regularization': 'l2', 'regularization_value': 0.0004, 'Units_hidden_layer0': 70, 'Units_hidden_layer1': 90, 'optimizer': 'adam', 'learning_rate': 0.0005, 'Units_hidden_layer2': 70}


## Hyperparameters tuning with **batching** (false attempt)

batching is not a learnable parameter. So a manual testing of batch number with fixed estimated params, might deliver better result

In [ ]:
class custom_HyperModel(keras_tuner.HyperModel):
    def build(self, hp:keras_tuner.HyperParameters) -> tf.keras.Model:
        # Activation
        curr_activation = hp.Choice('Activation', ['linear','relu', 'tanh', 'sigmoid'])
        
        # Dropout rate
        curr_dropout_rate = hp.Float('Dropout_rate',
                                        min_value = 0.0,
                                        max_value = 0.5,
                                        step = 0.05)
        
        # Regularization
        curr_regularization = hp.Choice('Regularization', ['l1', 'l2', 'l1_l2'])
        curr_regularization_value = hp.Float('regularization_value',   
                                            min_value = 0.0001,
                                            max_value = 0.001,
                                            step = 0.0001)
        
        if curr_regularization == 'l1':
            regularizer = keras.regularizers.L1(l1=curr_regularization_value)
        elif curr_regularization == 'l2':
            regularizer = keras.regularizers.L2(l2=curr_regularization_value)
        else:
            regularizer = keras.regularizers.L1L2(l1=curr_regularization_value, 
                                                l2=curr_regularization_value)

        model = keras.Sequential()

        max_hidden_layers = 3
        keep_sampling_hidden_units = True
        nr_hidden_layer = 0

        while keep_sampling_hidden_units:
            
            # Unit for hidden layer
            if nr_hidden_layer == 0:
                current_hidden_units = hp.Int('Units_hidden_layer{}'.format(nr_hidden_layer),
                                            min_value = 10,
                                            max_value = 200,
                                            step = 10)
            else:
                current_hidden_units = hp.Int('Units_hidden_layer{}'.format(nr_hidden_layer),
                                            min_value = 0,
                                            max_value = 200,
                                            step = 10)
            
            
            if current_hidden_units != 0:
                model.add(keras.layers.Dense(units = current_hidden_units,
                                            kernel_regularizer=regularizer,
                                            activation = curr_activation))
                
                model.add(keras.layers.Dropout(rate = curr_dropout_rate))

            if current_hidden_units == 0 or nr_hidden_layer == max_hidden_layers-1:
                keep_sampling_hidden_units = False
                
                
            
            nr_hidden_layer += 1
        
        model.add(keras.layers.Dense(units = 1,
                                    activation = 'linear'))
        

        # Optimizer and learning rate
        current_optimizer = hp.Choice('optimizer', ['adam', 'sgd'])
        current_learning_rate = hp.Float('learning_rate',
                                        min_value = 0.0001,
                                        max_value = 0.001,
                                        step = 0.0001)
        model.compile(optimizer = (keras.optimizers.Adam(learning_rate = current_learning_rate) if current_optimizer == 'adam' else keras.optimizers.SGD(learning_rate = current_learning_rate)),
                                loss = 'mse')
        
        return model
    
    def fit(self, hp:keras_tuner.HyperParameters, model:tf.keras.Model, *args, **kwargs):
        batch_size = hp.Choice('batch_size', [16, 32, 64, 128, 256])
        
        return model.fit(
                    *args,
                    batch_size=batch_size,
                    **kwargs
                )
    

es_callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    mode='min')

tuner = keras_tuner.RandomSearch(
    custom_HyperModel(),
    objective = 'val_loss',
    max_trials = 100,
    directory = './tmp/checkpoint',
    overwrite = True)

tuner.search(
    x_train_scaled, 
    y_train_scaled, 
    epochs = 100, 
    validation_data = (x_val_scaled, y_val_scaled), 
    callbacks = [es_callback])


best_model = tuner.get_best_models()[0]

print('Best hyperparameters: {}'.format(tuner.get_best_hyperparameters()[0].values))

## KFold averaged Hyperparameter tuning (false attempt)

k-folding in the tuning creates a nested cross-validation, while not necessarily improves the tuning

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

val_losses = []

x_train_raw_array = x_train_raw.to_numpy()
y_train_raw_array = y_train_raw.to_numpy()

es_callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    mode='min')


for fold, (train_idx, val_idx) in enumerate(kf.split(x_train_raw)):
    print("runnig {} fold".format(fold+1))

    x_train, x_val = x_train_raw_array[train_idx], x_train_raw_array[val_idx]
    y_train, y_val = y_train_raw_array[train_idx], y_train_raw_array[val_idx]

    x_scaler = StandardScaler()
    x_scaler.fit(x_train)

    x_train_scaled, x_val_scaled = x_scaler.transform(x_train), x_scaler.transform(x_val)


    y_scaler = StandardScaler()
    y_scaler.fit(y_train)

    y_train_scaled, y_val_scaled = y_scaler.transform(y_train), y_scaler.transform(y_val)


    tuner = keras_tuner.RandomSearch(
        keras_hp_optimization,
        objective = 'val_loss',
        max_trials = 10,
        directory = f'./tmp/checkpoint_fold{fold}',
        overwrite = True)

    tuner.search(
        x_train_scaled, 
        y_train_scaled, 
        epochs = 100, 
        validation_data = (x_val_scaled, y_val_scaled), 
        callbacks = [es_callback])

    best_model = tuner.get_best_models()[0]
    val_loss = best_model.evaluate(x_val_scaled, y_val_scaled, verbose=1)
    val_losses.append(val_loss)

print("Average val MSE across folds:", np.mean(val_losses))

print('Best hyperparameters: {}'.format(tuner.get_best_hyperparameters()[0].values))



Trial 10 Complete [00h 00m 06s]
val_loss: 1.8557032346725464

Best val_loss So Far: 0.023068025708198547
Total elapsed time: 00h 01m 02s
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0230 
Average val MSE across folds: 0.03404945181682706
Best hyperparameters: {'Activation': 'tanh', 'Dropout_rate': 0.0, 'Regularization': 'l1_l2', 'regularization_value': 0.0005, 'Units_hidden_layer0': 80, 'Units_hidden_layer1': 130, 'optimizer': 'adam', 'learning_rate': 0.0004, 'Units_hidden_layer2': 10}


/Users/benedikt/Documents/Master/SoSe 25/Deep Learning for Text Analytics/Coding/venv/lib/python3.9/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## Main (tuned)

In [ ]:
# Tuning n_units(hidden layer), dropout_rate, and activation function
tf_model = build_tf_model(n_features = (x_train_scaled.shape[1], ),
                          list_units = [100, 50, 25],
                          dropout_rate = 0.3,
                          regularization= [['l1', 2]],
                          activation = 'relu')

# Tuning learning rate, optimizer and epochs
tf_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
n_epochs = 100

train_loss = []
val_loss = []

# Early stoppping criterium
patience = 10
wait = 2
# We would like to minimize our loss, thus we set the initial best value to inf+
best = float('inf')

for epoch in range(n_epochs):
    epoch_train_loss = []
    epoch_val_loss = []

    '''
    Applying k-fold cross-validation here!!
    (https://www.geeksforgeeks.org/cross-validation-using-k-fold-with-scikit-learn/#kfold-with-scikitlearn)
    '''

    # Training
    for (x_batch_train,y_batch_train) in tf_train_rescaled:
        curr_train_loss = training(model = tf_model,
                              x_batch_data = x_batch_train,
                              y_batch_data = y_batch_train,
                              optimizer = tf_optimizer)

        epoch_train_loss.append(curr_train_loss.numpy())

    train_loss.append(np.mean(epoch_train_loss))

    # Validation 
    for (x_batch_val,y_batch_val) in tf_val_rescaled:

        predictions_val = tf_model(x_batch_val)
        
        curr_val_loss = tf.keras.losses.MSE(y_pred = tf.squeeze(predictions_val,axis=-1),
                                            y_true = tf.squeeze(y_batch_val,axis=-1))
        
        epoch_val_loss.append(curr_val_loss.numpy())

    epoch_val_loss = np.mean(epoch_val_loss)
    val_loss.append(epoch_val_loss)

    #Early stopping callback:
    wait += 1

    if epoch_val_loss < best:
      best = epoch_val_loss
      wait = 0
    if wait >= patience:
      break

print('MSE training: {} and compared to MSE validation: {}'.format(min(val_loss), min(train_loss)))


#  Visualization

In [ ]:
# Visualize the model performance:
plt.figure(figsize=(7,5))
plt.plot(train_loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.ylabel('MSE Score')
plt.xlabel('Epoch')
plt.title('Train vs. Validation Performance')
plt.legend()
plt.show()
plt.close()

In [ ]:
def keras_hp_optimization(hp:keras_tuner.HyperParameters)->keras.Sequential:

    # Activation
    curr_activation = hp.Choice('Activation', ['linear','relu', 'tanh', 'sigmoid'])
    
    # Regularization
    curr_regularization = hp.Choice('Regularization', ['l1', 'l2', 'l1_l2'])
    curr_regularization_value = hp.Float('regularization_value',   
                                        min_value = 0.0001,
                                        max_value = 0.001,
                                        step = 0.0001)
    
    if curr_regularization == 'l1':
        regularizer = keras.regularizers.L1(l1=curr_regularization_value)
    elif curr_regularization == 'l2':
        regularizer = keras.regularizers.L2(l2=curr_regularization_value)
    else:
        regularizer = keras.regularizers.L1L2(l1=curr_regularization_value, 
                                            l2=curr_regularization_value)

    model = keras.Sequential()

    max_hidden_layers = 4
    keep_sampling_hidden_units = True
    nr_hidden_layer = 0

    while keep_sampling_hidden_units:
        
        # Unit for hidden layer
        if nr_hidden_layer == 0:
            current_hidden_units = hp.Int('Units_hidden_layer{}'.format(nr_hidden_layer),
                                        min_value = 10,
                                        max_value = 200,
                                        step = 10)
        else:
            current_hidden_units = hp.Int('Units_hidden_layer{}'.format(nr_hidden_layer),
                                        min_value = 0,
                                        max_value = 200,
                                        step = 10)
        
        
        if current_hidden_units != 0:
            model.add(keras.layers.Dense(units = current_hidden_units,
                                        kernel_regularizer=regularizer,
                                        activation = curr_activation))

        if current_hidden_units == 0 or nr_hidden_layer == max_hidden_layers-1:
            keep_sampling_hidden_units = False
            
            
        
        nr_hidden_layer += 1
    
    model.add(keras.layers.Dense(units = 1,
                                activation = 'linear'))
    

    # Optimizer and learning rate
    current_optimizer = hp.Choice('optimizer', ['adam', 'sgd'])
    current_learning_rate = hp.Float('learning_rate',
                                    min_value = 0.0001,
                                    max_value = 0.001,
                                    step = 0.0001)
    model.compile(optimizer = (keras.optimizers.Adam(learning_rate = current_learning_rate) if current_optimizer == 'adam' else keras.optimizers.SGD(learning_rate = current_learning_rate)),
                            loss = 'mse')
    
    return model




es_callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    mode='min')

tuner = keras_tuner.RandomSearch(keras_hp_optimization,
                                 objective = 'val_loss',
                                 max_trials = 100,
                                 directory = './tmp/checkpoint',
                                 overwrite = True)


tuner.search(x_train_scaled, 
             y_train_scaled, 
             epochs = 100, 
             validation_data = (x_val_scaled, y_val_scaled), 
             callbacks = [es_callback])

best_model = tuner.get_best_models()[0]

print('Best hyperparameters: {}'.format(tuner.get_best_hyperparameters()[0].values))



Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
linear            |linear            |Activation
l1                |l1                |Regularization
0.0002            |0.0002            |regularization_value
160               |160               |Units_hidden_layer0
50                |50                |Units_hidden_layer1
sgd               |sgd               |optimizer
0.0006            |0.0006            |learning_rate

Epoch 1/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6487 - val_loss: 0.4595
Epoch 2/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4313 - val_loss: 0.3839
Epoch 3/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 941us/step - loss: 0.3634 - val_loss: 0.3405
Epoch 4/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 928us/step - loss: 0.3232 - val_loss: 0.3115
Epoch 5/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 937us/step - loss: 0.2935 - val_loss: 0.2904
Epoch 6/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 982us/step - loss: 0.2783 - val_loss: 0.2759
Epoch 7/100
47/47 ━━━━━━